In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, d=1, norm=True):
        super().__init__()
        layers = [
            nn.Conv2d(
                in_ch, out_ch,
                kernel_size=k,
                stride=s,
                padding=p,
                dilation=d,
                bias=not norm
            )
        ]
        if norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.1, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class ContextRefinementNet(nn.Module):
    """
    PWC-style context network with dilated convolutions.
    Input:
        feat: [B, C, H, W]
    Output:
        refined residual flow: [B, 2, H, W]
    """
    def __init__(self, in_ch):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(in_ch, 128, k=3, p=1, d=1),
            ConvBlock(128, 128, k=3, p=2, d=2),
            ConvBlock(128, 128, k=3, p=4, d=4),
            ConvBlock(128, 96,  k=3, p=8, d=8),
            ConvBlock(96, 64,   k=3, p=16, d=16),
            ConvBlock(64, 32,   k=3, p=1, d=1),
            nn.Conv2d(32, 2, kernel_size=3, stride=1, padding=1)
        )

    def forward(self, x):
        return self.net(x)


class PWCStyleDecoder(nn.Module):
    """
    Simple decoder according to your design:
      latent -> conv refinement -> coarse flow -> context refinement -> upsample

    Input:
        latent: [B, latent_ch, Hc, Wc]
    Output:
        flow_coarse:  [B, 2, Hc, Wc]
        flow_refined: [B, 2, Hc, Wc]
        flow_full:    [B, 2, H, W]
    """
    def __init__(self, latent_ch=128, hidden_ch=128):
        super().__init__()

        self.decode_feat = nn.Sequential(
            ConvBlock(latent_ch, hidden_ch, k=3, s=1, p=1),
            ConvBlock(hidden_ch, hidden_ch, k=3, s=1, p=1),
            ConvBlock(hidden_ch, 96, k=3, s=1, p=1),
            ConvBlock(96, 64, k=3, s=1, p=1),
        )

        self.flow_head = nn.Conv2d(64, 2, kernel_size=3, stride=1, padding=1)

        # context net takes decoder features + coarse flow
        self.context_net = ContextRefinementNet(in_ch=64 + 2)

    def forward(self, latent, out_size=None):
        """
        latent:   [B, C, Hc, Wc]
        out_size: (H, W) of final image resolution, e.g. (352, 1216)

        returns dict
        """
        feat = self.decode_feat(latent)                 # [B, 64, Hc, Wc]
        flow_coarse = self.flow_head(feat)              # [B, 2, Hc, Wc]

        context_in = torch.cat([feat, flow_coarse], dim=1)
        flow_residual = self.context_net(context_in)    # [B, 2, Hc, Wc]

        flow_refined = flow_coarse + flow_residual

        if out_size is not None:
            H, W = out_size
            flow_full = F.interpolate(
                flow_refined, size=(H, W),
                mode="bilinear", align_corners=False
            )

            # scale flow values because spatial resolution changed
            scale_y = H / flow_refined.shape[-2]
            scale_x = W / flow_refined.shape[-1]
            flow_full[:, 0] *= scale_x
            flow_full[:, 1] *= scale_y
        else:
            flow_full = flow_refined

        return {
            "decoder_feat": feat,
            "flow_coarse": flow_coarse,
            "flow_refined": flow_refined,
            "flow_full": flow_full,
        }